In [0]:
API_key = "JGtMTGvLjCWp1RHRJRa3F3xS"

Bronze layer

In [0]:


# %pip install requests   # run once if needed

import requests
from pyspark.sql import SparkSession

# 🔹 Spark Session
spark = SparkSession.builder.appName("JobETL_Bronze").getOrCreate()

# 🔹 API KEY (replace)
API_KEY = "1601617353445414c6c423156f34046232249c9c17051d398e1e3ae337885938"

# 🔹 Inputs (use safe values)
roles = ["Software Engineer", "Data Engineer"]
location = "Bangalore, India"

all_jobs = []

# 🔹 Loop
for role in roles:
    params = {
        "engine": "google_jobs",
        "q": role,
        "location": location,
        "api_key": API_KEY
    }

    res = requests.get("https://serpapi.com/search.json", params=params)

    print(f"\n🔎 Fetching for: {role}")
    print("Status Code:", res.status_code)

    data = res.json()
    print("Response Keys:", data.keys())   # debug

    # 🔴 IMPORTANT DEBUG
    if "error" in data:
        print("❌ API ERROR:", data["error"])
        continue

    jobs = data.get("jobs_results", [])
    print(f"Jobs fetched: {len(jobs)}")

    for job in jobs:
        job["search_role"] = role

    all_jobs.extend(jobs)

# 🔹 CHECK DATA BEFORE SPARK
print("\n📊 Total jobs collected:", len(all_jobs))

if not all_jobs:
    print("❌ No data received → Check API key / params / location")
else:
    # ✅ Create DataFrame (NO sparkContext)
    bronze_df = spark.createDataFrame(all_jobs)

    # 🔹 Preview
    bronze_df.show(5)
    bronze_df.printSchema()

    # 🔹 Save as table
    bronze_df.write.format("delta") \
        .option("mergeSchema", "true") \
        .mode("append") \
        .saveAsTable("bronze_jobs_raw")

    print("✅ Data Written to Bronze Layer")

In [0]:
display(bronze_df)

Silver layer
Cleaning and strcturing

In [0]:
from pyspark.sql.functions import*

spark = SparkSession.builder.appName("JobETL_Silver").getOrCreate()

# read Bronze layer
df = spark.read.format("delta").load("dbfs:/user/hive/warehouse/bronze_jobs_raw")

# clean and Str

silver_df = bronze_df.selectExpr(
    "title",
    "company_name",
    "location",
    "description",
    "job_id",
    "detected_extensions.posted_at as posted_at",
    "search_role"
).dropna(subset=["title", "company_name", "location"])


In [0]:
display(silver_df)

In [0]:
silver_df = silver_df.withColumn("company_name",trim(upper(col("company_name"))))
silver_df=silver_df.dropna(subset=["posted_at"])
silver_df.write.format("delta").option("mergeSchema", "true").mode("overwrite").saveAsTable("silver_jobs")
display(silver_df)

Gold Layer